# **Algo Trading - Backtester**:

The hello world!

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("🤖 Initializing Trading Bot...")

# 1. FETCH THE DATA
ticker = "SPY"
print(f"📥 Downloading historical data for {ticker}...")
df = yf.download(ticker, start="2005-01-01", end="2025-01-01", progress=False)

# Flatten yfinance's multi-index columns if necessary
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# 2. BUILD THE "BRAIN" (The Technical Indicators)
print("🧠 Calculating Moving Averages...")
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['SMA_200'] = df['Close'].rolling(window=200).mean()

# Drop the first 200 days because we can't calculate a 200-day average without them
df.dropna(inplace=True)

# 3. GENERATE TRADING SIGNALS
# Logic: If SMA_50 > SMA_200, assign 1 (BUY/HOLD). Otherwise, assign 0 (SELL/CASH).
df['Signal'] = np.where(df['SMA_50'] > df['SMA_200'], 1, 0)

# 🚨 THE MOST IMPORTANT LINE IN QUANT PROGRAMMING 🚨
# We must shift the signal forward by 1 day.
# Why? Because you only know the 50-day crossed the 200-day at the END of the trading day.
# You cannot trade today based on today's closing price. You must trade TOMORROW.
df['Position'] = df['Signal'].shift(1)

# 4. CALCULATE PERFORMANCE (Backtesting)
print("🧮 Simulating trades over 20 years...")
# What did the market do each day?
df['Market_Daily_Return'] = df['Close'].pct_change()

# What did our bot do? (Market return multiplied by our Position: 1 or 0)
# If Position is 0, we were holding cash, so our return that day is 0%.
df['Strategy_Daily_Return'] = df['Market_Daily_Return'] * df['Position']

# Calculate cumulative growth over time (Compound interest)
df['Buy_and_Hold_Growth'] = (1 + df['Market_Daily_Return']).cumprod()
df['Strategy_Growth'] = (1 + df['Strategy_Daily_Return']).cumprod()

# 5. THE RESULTS
buy_and_hold_final = (df['Buy_and_Hold_Growth'].iloc[-1] - 1) * 100
strategy_final = (df['Strategy_Growth'].iloc[-1] - 1) * 100

print("\n" + "="*40)
print("📊 BACKTEST RESULTS (2005 - 2025)")
print("="*40)
print(f"Total Return if you just held {ticker}: +{buy_and_hold_final:.2f}%")
print(f"Total Return of the Trading Bot:  +{strategy_final:.2f}%")

# Bonus: Did the bot protect us from crashes?
# Let's look at the maximum drawdown (the biggest drop from an all-time high)
df['Market_Peak'] = df['Buy_and_Hold_Growth'].cummax()
df['Market_Drawdown'] = (df['Buy_and_Hold_Growth'] - df['Market_Peak']) / df['Market_Peak']

df['Strategy_Peak'] = df['Strategy_Growth'].cummax()
df['Strategy_Drawdown'] = (df['Strategy_Growth'] - df['Strategy_Peak']) / df['Strategy_Peak']

print(f"\n📉 Worst Market Crash (Drawdown): {df['Market_Drawdown'].min() * 100:.2f}%")
print(f"📉 Worst Bot Crash (Drawdown):    {df['Strategy_Drawdown'].min() * 100:.2f}%")
print("="*40)

🤖 Initializing Trading Bot...
📥 Downloading historical data for SPY...
🧠 Calculating Moving Averages...
🧮 Simulating trades over 20 years...

📊 BACKTEST RESULTS (2005 - 2025)
Total Return if you just held SPY: +608.25%
Total Return of the Trading Bot:  +477.85%

📉 Worst Market Crash (Drawdown): -55.19%
📉 Worst Bot Crash (Drawdown):    -33.72%
